# Tail-Class Selective Background Inpainting Augmentation

이 노트북은 군용 항공기 YOLO 데이터셋에서 long-tail class에만 bbox-protected diffusion background inpainting을 적용하는 실험 파이프라인입니다. 기본값은 빠른 검증을 위한 `RUN_MODE = "smoke"`입니다.

## 1. Check GPU and environment

GPU, Python, CUDA 상태를 확인합니다.

In [ ]:
import os, sys, platform, subprocess
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch 확인 전 dependency 설치가 필요할 수 있습니다:", exc)
!nvidia-smi || true

## 2. Install dependencies

저장소 루트로 이동한 뒤 필요한 패키지를 설치합니다.

In [ ]:
from pathlib import Path
import os

if Path('/content/military-aircraft-tail-inpainting').exists():
    REPO_ROOT = Path('/content/military-aircraft-tail-inpainting')
elif Path.cwd().name == 'notebooks':
    REPO_ROOT = Path.cwd().parent
else:
    REPO_ROOT = Path.cwd()
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)

RUN_MODE = 'smoke'  # 'smoke' 또는 'full'
CONFIG = f'configs/{RUN_MODE}.yaml'
print('Config:', CONFIG)

In [ ]:
!pip -q install -r requirements.txt

## 3. Configure Kaggle API credentials

`kaggle.json`을 Colab에 업로드하거나 Google Drive에서 복사합니다.

In [ ]:
from pathlib import Path

# 방법 A: Colab 파일 업로드 후 /content/kaggle.json 위치에 둡니다.
# 방법 B: Google Drive에서 복사합니다. 예:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/kaggle.json /content/kaggle.json

if Path('/content/kaggle.json').exists():
    !mkdir -p ~/.kaggle
    !cp /content/kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json
    print('Kaggle credential configured.')
else:
    print('아직 /content/kaggle.json이 없습니다. 업로드 또는 Drive 복사 후 이 셀을 다시 실행하세요.')

## 4. Download Kaggle dataset

원본 데이터는 `/content/data/raw`에 저장하며, 기존 파일이 있으면 기본적으로 건너뜁니다.

In [ ]:
!python src/data/download_kaggle.py \
  --dataset rookieengg/military-aircraft-detection-dataset-yolo-format \
  --out /content/data/raw

## 5. Inspect dataset structure

데이터셋의 split, images/labels 경로, YAML 및 class names를 탐색합니다.

In [ ]:
!python src/data/inspect_dataset.py \
  --root /content/data/raw \
  --out /content/outputs/analysis/dataset_inspection.json

## 6. Normalize dataset to Ultralytics YOLO format

구조가 다른 YOLO 데이터셋을 `/content/data/processed/base` 아래 표준 구조로 정규화합니다.

In [ ]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(Path(CONFIG).read_text())
mode = cfg['mode']
max_images = mode.get('max_images_per_split')
max_classes = mode.get('max_classes')
cmd = 'python src/data/normalize_yolo_dataset.py --raw /content/data/raw --out /content/data/processed/base'
if max_images is not None:
    cmd += f' --max-images-per-split {max_images}'
if max_classes is not None:
    cmd += f' --max-classes {max_classes}'
print(cmd)
!{cmd}

## 7. Analyze class imbalance

class-wise instance/image count, bbox area, head/medium/tail group을 계산합니다.

In [ ]:
!python src/data/analyze_long_tail.py \
  --data /content/data/processed/base/data.yaml \
  --config {CONFIG} \
  --outputs /content/outputs

## 8. Train real-only YOLO baseline

baseline AP를 얻기 위해 원본 학습 데이터만 사용합니다.

In [ ]:
import yaml, subprocess
from pathlib import Path

cfg = yaml.safe_load(Path(CONFIG).read_text())
for seed in cfg['detector'].get('seeds', [42]):
    cmd = [
        'python', 'src/train/train_yolo.py',
        '--data', '/content/data/processed/base/data.yaml',
        '--config', CONFIG,
        '--name', 'real_only',
        '--seed', str(seed),
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)

## 9. Compute per-class AP and define head/medium/tail classes

가장 최근 real-only run의 best weight를 평가하고, baseline AP를 반영해 selective generation plan을 다시 생성합니다.

In [ ]:
from pathlib import Path
import yaml, subprocess

cfg = yaml.safe_load(Path(CONFIG).read_text())
for seed in cfg['detector'].get('seeds', [42]):
    runs = sorted(Path('/content/outputs/runs').glob(f'real_only_*_seed{seed}_*'), key=lambda p: p.stat().st_mtime)
    assert runs, f'real_only seed {seed} run을 찾지 못했습니다.'
    run_dir = runs[-1]
    weights = run_dir / 'weights' / 'best.pt'
    print('Run:', run_dir)
    print('Weights:', weights)
    cmd = [
        'python', 'src/eval/collect_yolo_metrics.py',
        '--run-dir', str(run_dir),
        '--outputs', '/content/outputs',
        '--experiment', 'real_only',
        '--seed', str(seed),
        '--model-name', cfg['detector']['model'],
        '--data', '/content/data/processed/base/data.yaml',
        '--weights', str(weights),
        '--config', CONFIG,
    ]
    subprocess.run(cmd, check=True)

subprocess.run([
    'python', 'src/data/analyze_long_tail.py',
    '--data', '/content/data/processed/base/data.yaml',
    '--config', CONFIG,
    '--outputs', '/content/outputs',
    '--baseline-ap', '/content/outputs/metrics/per_class_ap.csv',
], check=True)

## 10. Generate bbox-protected background inpainting images for selected tail classes

항공기 bbox 내부는 보호하고 배경만 diffusion inpainting합니다. Smoke mode도 시간이 걸릴 수 있습니다.

In [ ]:
!python src/augment/inpaint_background.py \
  --data /content/data/processed/base/data.yaml \
  --plan /content/outputs/analysis/augmentation_plan_uniform.csv \
  --config {CONFIG} \
  --out /content/data/processed/synthetic_inpaint \
  --outputs /content/outputs \
  --plan-name uniform

!python src/augment/inpaint_background.py \
  --data /content/data/processed/base/data.yaml \
  --plan /content/outputs/analysis/augmentation_plan_selective.csv \
  --config {CONFIG} \
  --out /content/data/processed/synthetic_inpaint \
  --outputs /content/outputs \
  --plan-name selective

## 11. Build experimental datasets

A-E 실험군을 만들고, validation/test split은 모든 실험군에서 동일하게 유지합니다.

In [ ]:
!python src/augment/build_experiment_datasets.py \
  --base-data /content/data/processed/base/data.yaml \
  --config {CONFIG} \
  --experiments-root /content/data/experiments \
  --uniform-plan /content/outputs/analysis/augmentation_plan_uniform.csv \
  --selective-plan /content/outputs/analysis/augmentation_plan_selective.csv \
  --synthetic-root /content/data/processed/synthetic_inpaint

## 12. Train all experiment groups

Smoke mode는 짧은 학습으로 파이프라인 검증을 목표로 합니다. Full mode는 논문용 결과 산출에 사용합니다.

In [ ]:
import yaml, subprocess
from pathlib import Path

cfg = yaml.safe_load(Path(CONFIG).read_text())
for variant in cfg['experiments']['variants']:
    if variant == 'real_only':
        continue
    data_yaml = Path('/content/data/experiments') / variant / 'data.yaml'
    for seed in cfg['detector'].get('seeds', [42]):
        cmd = [
            'python', 'src/train/train_yolo.py',
            '--data', str(data_yaml),
            '--config', CONFIG,
            '--name', variant,
            '--seed', str(seed),
        ]
        if variant == 'basic_aug':
            cmd.append('--basic-aug')
        print(' '.join(cmd))
        subprocess.run(cmd, check=True)

## 13. Collect metrics

각 실험군의 가장 최근 run에서 per-class AP와 overall metric을 수집합니다.

In [ ]:
import yaml, subprocess
from pathlib import Path

cfg = yaml.safe_load(Path(CONFIG).read_text())
for variant in cfg['experiments']['variants']:
    for seed in cfg['detector'].get('seeds', [42]):
        runs = sorted(Path('/content/outputs/runs').glob(f'{variant}_*_seed{seed}_*'), key=lambda p: p.stat().st_mtime)
        if not runs:
            print('skip, no run:', variant, seed)
            continue
        run_dir = runs[-1]
        weights = run_dir / 'weights' / 'best.pt'
        data_yaml = Path('/content/data/processed/base/data.yaml') if variant == 'real_only' else Path('/content/data/experiments') / variant / 'data.yaml'
        cmd = [
            'python', 'src/eval/collect_yolo_metrics.py',
            '--run-dir', str(run_dir),
            '--outputs', '/content/outputs',
            '--experiment', variant,
            '--seed', str(seed),
            '--model-name', cfg['detector']['model'],
            '--data', str(data_yaml),
            '--weights', str(weights),
            '--config', CONFIG,
        ]
        print(' '.join(cmd))
        subprocess.run(cmd, check=True)

subprocess.run([
    'python', 'src/eval/compute_long_tail_metrics.py',
    '--raw', '/content/outputs/metrics/raw_yolo_metrics.csv',
    '--per-class', '/content/outputs/metrics/per_class_ap.csv',
    '--groups', '/content/outputs/analysis/class_groups.csv',
    '--outputs', '/content/outputs',
], check=True)

## 14. Plot results

논문 표/그림 후보가 되는 비교 plot을 생성합니다.

In [ ]:
!python src/eval/plot_results.py --outputs /content/outputs
from pathlib import Path
print(sorted(Path('/content/outputs/figures').glob('*.png')))

## 15. Save outputs to Google Drive

Colab 세션 종료에 대비해 결과물을 Drive에 복사합니다.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/military-aircraft-tail-inpainting-outputs
# !rsync -av /content/outputs/ /content/drive/MyDrive/military-aircraft-tail-inpainting-outputs/
print('필요하면 위 주석을 해제해 Google Drive에 저장하세요.')

## Optional: one-command pipeline

위 단계를 한 번에 실행하려면 아래 명령을 사용합니다.

In [ ]:
# 빠른 smoke test
# !python src/run_pipeline.py --config configs/smoke.yaml

# 논문용 full experiment
# !python src/run_pipeline.py --config configs/full.yaml